# Loan Default Risk Prediction
3D Club Induction Task 2026 - Task 1

## 1. Problem Statement
Banks lose money when they approve loans that are never repaid. The goal is to predict whether a borrower will experience **serious delinquency (90+ days past due) within two years** (`SeriousDlqin2yrs = 1`), so risky applications can be flagged early.

Only a small fraction of borrowers default (~6.7%), so this is an **imbalanced binary classification** problem. A model that predicts "nobody defaults" would reach ~93% accuracy and be useless. We therefore:
- use **PR-AUC (average precision)** and **ROC-AUC** for model selection, not accuracy,
- handle imbalance with **class weighting** (applied only to training data),
- choose the **decision threshold** explicitly instead of defaulting to 0.5,
- report precision, recall, F1 and a confusion matrix on a held-out test set.

## 2. Dataset Description
Kaggle competition *Give Me Some Credit* (`cs-training.csv`, 150,000 borrowers). Download it from Kaggle and place it next to this notebook (or edit `DATA_PATH`).

| Column | Meaning |
|---|---|
| SeriousDlqin2yrs | **Target**: 90+ days past due within 2 years |
| RevolvingUtilizationOfUnsecuredLines | Balance on cards/credit lines divided by credit limits |
| age | Borrower age |
| NumberOfTime30-59DaysPastDueNotWorse | Times 30-59 days late |
| DebtRatio | Monthly debt payments / monthly gross income |
| MonthlyIncome | Monthly income |
| NumberOfOpenCreditLinesAndLoans | Open loans and credit lines |
| NumberOfTimes90DaysLate | Times 90+ days late |
| NumberRealEstateLoansOrLines | Mortgage and real estate loans |
| NumberOfTime60-89DaysPastDueNotWorse | Times 60-89 days late |
| NumberOfDependents | Dependents (excluding self) |

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve, roc_curve,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)
from sklearn.inspection import permutation_importance

SEED = 42
np.random.seed(SEED)
sns.set_style("whitegrid")
DATA_PATH = "cs-training.csv"

## 3. Data Loading and Cleaning

In [ ]:
df = pd.read_csv(DATA_PATH)
if "Unnamed: 0" in df.columns:
    df = df.drop(columns="Unnamed: 0")
print(df.shape)
df.head()

In [ ]:
print("Duplicates:", df.duplicated().sum())
print(df.isna().sum()[lambda s: s > 0])
df.describe().T

Problems found and how we fix them:
- **Missing values**: `MonthlyIncome` (~20%) and `NumberOfDependents` (~2.6%). Median imputation, plus a missing-income flag because missingness itself can be informative. (Imputation statistics are computed from the training split only, to avoid leakage.)
- **Impossible age**: age 0 is invalid, so we drop those rows.
- **Sentinel codes**: the three late-payment columns contain 96 and 98, which are placeholder codes, not real counts. We replace them with NaN, then impute with the median (and flag).
- **Extreme outliers**: `RevolvingUtilizationOfUnsecuredLines` and `DebtRatio` have enormous values, so we clip them at the 99.5th percentile (computed on training data).

In [ ]:
df = df.drop_duplicates()
df = df[df["age"] > 0].copy()

late_cols = ["NumberOfTime30-59DaysPastDueNotWorse",
             "NumberOfTimes90DaysLate",
             "NumberOfTime60-89DaysPastDueNotWorse"]
df["sentinel_late_flag"] = df[late_cols].isin([96, 98]).any(axis=1).astype(int)
df[late_cols] = df[late_cols].replace({96: np.nan, 98: np.nan})
print(df.shape, "| default rate:", round(df["SeriousDlqin2yrs"].mean(), 4))

## 4. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df["SeriousDlqin2yrs"].value_counts().plot.bar(ax=ax[0], color=["#4c72b0", "#c44e52"])
ax[0].set_title("Class distribution"); ax[0].set_xticklabels(["No default (0)", "Default (1)"], rotation=0)
sns.heatmap(df.corr(numeric_only=True)[["SeriousDlqin2yrs"]].sort_values("SeriousDlqin2yrs", ascending=False),
            annot=True, fmt=".2f", cmap="coolwarm", ax=ax[1], cbar=False)
ax[1].set_title("Correlation with target")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
sns.boxplot(data=df, x="SeriousDlqin2yrs", y="age", ax=axes[0,0])
sns.boxplot(data=df, x="SeriousDlqin2yrs", y=df["MonthlyIncome"].clip(upper=df["MonthlyIncome"].quantile(.95)), ax=axes[0,1])
axes[0,1].set_ylabel("MonthlyIncome (clipped at 95th pct)")
sns.boxplot(data=df, x="SeriousDlqin2yrs", y=df["RevolvingUtilizationOfUnsecuredLines"].clip(upper=2), ax=axes[0,2])
axes[0,2].set_ylabel("Revolving utilization (clipped at 2)")
for a, c in zip(axes[1], late_cols):
    rate = df.groupby(df[c].clip(upper=6))["SeriousDlqin2yrs"].mean()
    rate.plot.bar(ax=a, color="#c44e52"); a.set_title(f"Default rate by\n{c} (capped at 6)"); a.set_ylabel("default rate")
plt.tight_layout(); plt.show()

**Observations to write up after running:** past-due counts and high revolving utilization are strongly associated with default; younger borrowers default more; income shows a weaker, noisier relationship. Check the plots above and add your own comments.

## 5. Feature Engineering
New features built from domain knowledge (all row-wise, so no leakage):
- `TotalPastDue`: sum of all late-payment counts
- `AnyPastDue` / `Any90Late`: binary indicators
- `IncomePerPerson`: income divided by (dependents + 1)
- `MonthlyDebt`: DebtRatio x income (approximate absolute debt payment)
- `AgeBucket` is skipped on purpose; tree models and `age` already capture it.
- `log1p` transforms of skewed features are applied inside the scaled logistic regression pipeline only via the engineered log columns.

In [ ]:
def add_features(d):
    d = d.copy()
    d["TotalPastDue"] = d[late_cols].sum(axis=1)
    d["AnyPastDue"] = (d["TotalPastDue"] > 0).astype(int)
    d["Any90Late"] = (d["NumberOfTimes90DaysLate"] > 0).astype(int)
    d["IncomePerPerson"] = d["MonthlyIncome"] / (d["NumberOfDependents"] + 1)
    d["MonthlyDebt"] = d["DebtRatio"] * d["MonthlyIncome"]
    d["log_Income"] = np.log1p(d["MonthlyIncome"])
    d["log_Utilization"] = np.log1p(d["RevolvingUtilizationOfUnsecuredLines"])
    d["log_DebtRatio"] = np.log1p(d["DebtRatio"])
    return d

## 6. Train-Test Split and Imbalance Handling
We make a **stratified 80/20 split** first, then learn all cleaning statistics (medians, clip limits) from the **training set only**.

Imbalance strategy: **class weighting** (`class_weight="balanced"`) rather than SMOTE. It adds no synthetic points, has no leakage risk during cross-validation, and works for every model used here. All models are compared with PR-AUC, which is not distorted by the majority class.

In [ ]:
y = df["SeriousDlqin2yrs"]
X = df.drop(columns="SeriousDlqin2yrs")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

# statistics from TRAIN only
impute_cols = late_cols + ["MonthlyIncome", "NumberOfDependents"]
medians = X_train[impute_cols].median()
clip_hi = {c: X_train[c].quantile(0.995) for c in ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio", "MonthlyIncome"]}

def preprocess(d):
    d = d.copy()
    d["income_missing"] = d["MonthlyIncome"].isna().astype(int)
    d["dependents_missing"] = d["NumberOfDependents"].isna().astype(int)
    d[impute_cols] = d[impute_cols].fillna(medians)
    for c, hi in clip_hi.items():
        d[c] = d[c].clip(upper=hi)
    return add_features(d)

X_train_p, X_test_p = preprocess(X_train), preprocess(X_test)
print("Train:", X_train_p.shape, "default rate", round(y_train.mean(), 4))
print("Test: ", X_test_p.shape,  "default rate", round(y_test.mean(), 4))
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("Imbalance ratio (neg/pos):", round(pos_weight, 1))

## 7. Model Selection and Justification
| Model | Why |
|---|---|
| **Logistic Regression** (scaled) | Interpretable baseline, widely used in credit scoring |
| **Random Forest** | Captures non-linear interactions, robust to outliers |
| **Histogram Gradient Boosting** | Usually the strongest on tabular data, handles skew and non-linearity natively |

All use class weighting. Models are compared with **5-fold stratified CV on the training set**, ranked by **PR-AUC** (primary) and ROC-AUC.

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000)),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED),
}
rows = []
for name, m in models.items():
    p = cross_val_predict(m, X_train_p, y_train, cv=skf, method="predict_proba", n_jobs=1)[:, 1]
    rows.append({"model": name, "CV PR-AUC": average_precision_score(y_train, p), "CV ROC-AUC": roc_auc_score(y_train, p)})
cv_results = pd.DataFrame(rows).set_index("model").sort_values("CV PR-AUC", ascending=False)
cv_results.round(4)

## 8. Model Training and Hyperparameter Tuning
We tune the best family (gradient boosting) with `RandomizedSearchCV`, optimising **average precision** with stratified CV. Tuning uses the training set only.

In [ ]:
param_dist = {
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_iter": [200, 300, 500],
    "max_depth": [3, 4, 5, None],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [20, 50, 100],
    "l2_regularization": [0.0, 0.1, 1.0, 10.0],
}
search = RandomizedSearchCV(
    HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED),
    param_dist, n_iter=20, scoring="average_precision", cv=skf, n_jobs=-1, random_state=SEED, verbose=1)
search.fit(X_train_p, y_train)
print("Best params:", search.best_params_)
print("Best CV PR-AUC:", round(search.best_score_, 4))
best_model = search.best_estimator_

## 9. Threshold Selection
The default 0.5 threshold is arbitrary on imbalanced data. We pick the threshold using **out-of-fold predictions on the training set** (never the test set), then freeze it.

Business framing: missing a defaulter (false negative) costs far more than rejecting a good borrower (false positive), so we favour recall. We compare two rules:
1. the threshold maximising **F1**,
2. the highest threshold that keeps **recall >= 75%**.

We use rule 2 as the operating point; change `TARGET_RECALL` to suit a different risk appetite.

In [ ]:
oof = cross_val_predict(best_model, X_train_p, y_train, cv=skf, method="predict_proba", n_jobs=1)[:, 1]
prec, rec, thr = precision_recall_curve(y_train, oof)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
thr_f1 = thr[np.argmax(f1s)]

TARGET_RECALL = 0.75
ok = np.where(rec[:-1] >= TARGET_RECALL)[0]
thr_recall = thr[ok[-1]]

print(f"Max-F1 threshold: {thr_f1:.3f}  (precision {prec[np.argmax(f1s)]:.3f}, recall {rec[np.argmax(f1s)]:.3f})")
print(f"Recall>={TARGET_RECALL:.0%} threshold: {thr_recall:.3f}  (precision {prec[ok[-1]]:.3f}, recall {rec[ok[-1]]:.3f})")

plt.figure(figsize=(7, 4.5))
plt.plot(thr, prec[:-1], label="Precision"); plt.plot(thr, rec[:-1], label="Recall"); plt.plot(thr, f1s, label="F1")
plt.axvline(thr_f1, ls="--", c="gray", label=f"max-F1 ({thr_f1:.2f})")
plt.axvline(thr_recall, ls="--", c="red", label=f"recall target ({thr_recall:.2f})")
plt.xlabel("Threshold"); plt.title("Precision / Recall / F1 vs threshold (out-of-fold)"); plt.legend(); plt.show()

THRESHOLD = thr_recall

## 10. Model Evaluation
Final model is evaluated **once** on the untouched test set at the chosen threshold, and compared against the baseline models trained on the same data.

In [ ]:
best_model.fit(X_train_p, y_train)
proba = best_model.predict_proba(X_test_p)[:, 1]
pred = (proba >= THRESHOLD).astype(int)

print(f"ROC-AUC: {roc_auc_score(y_test, proba):.4f}")
print(f"PR-AUC : {average_precision_score(y_test, proba):.4f}  (random baseline = {y_test.mean():.4f})")
print(f"Threshold: {THRESHOLD:.3f}")
print(f"Precision: {precision_score(y_test, pred):.3f} | Recall: {recall_score(y_test, pred):.3f} | F1: {f1_score(y_test, pred):.3f}\n")
print(classification_report(y_test, pred, target_names=["No default", "Default"]))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["No default", "Default"]).plot(ax=ax[0], cmap="Blues", colorbar=False)
ax[0].set_title(f"Confusion matrix @ threshold {THRESHOLD:.2f}")

fpr, tpr, _ = roc_curve(y_test, proba)
ax[1].plot(fpr, tpr, label=f"Tuned HGB (AUC {roc_auc_score(y_test, proba):.3f})"); ax[1].plot([0,1],[0,1],"k--")
ax[1].set_xlabel("False positive rate"); ax[1].set_ylabel("True positive rate"); ax[1].set_title("ROC curve"); ax[1].legend()

p, r, _ = precision_recall_curve(y_test, proba)
ax[2].plot(r, p, label=f"Tuned HGB (AP {average_precision_score(y_test, proba):.3f})")
ax[2].axhline(y_test.mean(), ls="--", c="k", label="No-skill")
ax[2].set_xlabel("Recall"); ax[2].set_ylabel("Precision"); ax[2].set_title("Precision-Recall curve"); ax[2].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Test-set comparison of all models
rows = []
for name, m in {**models, "Tuned HGB": best_model}.items():
    m = m if name == "Tuned HGB" else m.fit(X_train_p, y_train)
    pr = m.predict_proba(X_test_p)[:, 1]
    rows.append({"model": name, "ROC-AUC": roc_auc_score(y_test, pr), "PR-AUC": average_precision_score(y_test, pr)})
pd.DataFrame(rows).set_index("model").round(4)

**Interpretation (fill in with your numbers):** at the chosen threshold the model catches roughly the target share of defaulters, at the cost of flagging many non-defaulters (low precision is expected when the base rate is ~7%). A bank would pair this score with manual review of flagged cases. Accuracy is deliberately not reported as a headline metric.

## 11. Feature Importance

In [ ]:
perm = permutation_importance(best_model, X_test_p, y_test, scoring="average_precision",
                              n_repeats=10, random_state=SEED, n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X_test_p.columns).sort_values()
imp.tail(12).plot.barh(figsize=(8, 5), color="#4c72b0")
plt.xlabel("Drop in PR-AUC when feature is shuffled"); plt.title("Permutation importance (test set)")
plt.tight_layout(); plt.show()
imp.sort_values(ascending=False).head(10).round(4)

**Discussion:** past-due history features and revolving utilization are expected to dominate, matching the EDA. Optionally add SHAP (`pip install shap`; `shap.TreeExplainer`) for per-prediction explanations.

## Conclusion
- Class weighting + PR-AUC-based selection avoids the "predict nobody defaults" trap.
- Gradient boosting outperformed the logistic regression baseline (confirm in your results table).
- Threshold chosen on training out-of-fold predictions to hit a recall target, then evaluated once on the test set.
- Limitations: no cost-sensitive optimisation with real loan economics, no probability calibration, single dataset snapshot.